In [8]:
#First 5 images are UUVs and the second 5 are spinner dolfins 
labels =[0,0,0,0,0,1,1,1,1,1]


In [11]:
import glob
from PIL import Image
import numpy as np

# Update this to YOUR image folder
folder_path = r"C:\Users\yusle\Desktop\Git Repositories\OSINModels\TestUUV_IMG\*.png"

#Verify the Imaages are loading correctly 
for file_path in glob.glob(folder_path):
    print("File:", file_path)

    # Read image
    img = Image.open(file_path).convert("RGB")

    # Convert to numpy (optional)
    img_arr = np.array(img)

    print("Image shape:", img_arr.shape)

File: C:\Users\yusle\Desktop\Git Repositories\OSINModels\TestUUV_IMG\spectrogram1.png
Image shape: (480, 640, 3)
File: C:\Users\yusle\Desktop\Git Repositories\OSINModels\TestUUV_IMG\spectrogram10.png
Image shape: (480, 640, 3)
File: C:\Users\yusle\Desktop\Git Repositories\OSINModels\TestUUV_IMG\spectrogram2.png
Image shape: (480, 640, 3)
File: C:\Users\yusle\Desktop\Git Repositories\OSINModels\TestUUV_IMG\spectrogram3.png
Image shape: (480, 640, 3)
File: C:\Users\yusle\Desktop\Git Repositories\OSINModels\TestUUV_IMG\spectrogram4.png
Image shape: (480, 640, 3)
File: C:\Users\yusle\Desktop\Git Repositories\OSINModels\TestUUV_IMG\spectrogram5.png
Image shape: (480, 640, 3)
File: C:\Users\yusle\Desktop\Git Repositories\OSINModels\TestUUV_IMG\spectrogram6.png
Image shape: (480, 640, 3)
File: C:\Users\yusle\Desktop\Git Repositories\OSINModels\TestUUV_IMG\spectrogram7.png
Image shape: (480, 640, 3)
File: C:\Users\yusle\Desktop\Git Repositories\OSINModels\TestUUV_IMG\spectrogram8.png
Image sha

In [6]:
#creating dataset class
from torch.utils.data import Dataset
from PIL import Image
import glob

class ImageDataset(Dataset):
    def __init__(self, folder_path, labels, transform=None):
        self.image_paths = sorted(glob.glob(folder_path))  # sort for consistent ordering
        self.labels = labels
        self.transform = transform

        assert len(self.image_paths) == len(self.labels), \
            "Number of images and labels must match!"

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        img = Image.open(img_path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, self.labels[idx]


In [15]:
import glob

image_paths = sorted(glob.glob(r"C:\Users\yusle\Desktop\Git Repositories\OSINModels\TestUUV_IMG\*.png"))


In [30]:
from sklearn.model_selection import train_test_split

# First split train vs temp (temp = val + test)
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    image_paths, labels, test_size=0.40, random_state=42, stratify=labels
)

# Now split temp into val and test
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.50, random_state=42, stratify=temp_labels
)

print("Train:", len(train_paths))
print("Val:", len(val_paths))
print("Test:", len(test_paths))
print("Train labels:",len(train_labels))
print("Val labels: ",len(val_labels))
print("Test labels: ",len(test_labels))


Train: 6
Val: 2
Test: 2
Train labels: 6
Val labels:  2
Test labels:  2


In [18]:
from torch.utils.data import Dataset
from PIL import Image

class ImageDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, self.labels[idx]


In [ ]:
#Transform and dataloader 
import torch
from torchvision import transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor()
])

train_dataset = ImageDataset(train_paths, train_labels, transform)
val_dataset   = ImageDataset(val_paths,  val_labels,  transform)
test_dataset  = ImageDataset(test_paths, test_labels, transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32)
test_loader  = DataLoader(test_dataset,  batch_size=32)


In [20]:
#Building simple CNN 
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()

        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(64 * 16 * 16, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        x = x.view(x.size(0), -1)

        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


In [33]:
#Set up for training 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

num_classes = len(set(train_labels))  # only count classes in training set
model = SimpleCNN(num_classes=num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 10

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    # -------------------- TRAINING --------------------
    for images, lbls in train_loader:
        images, lbls = images.to(device), lbls.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, lbls)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)

    # -------------------- VALIDATION --------------------
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, lbls in val_loader:
            images, lbls = images.to(device), lbls.to(device)

            outputs = model(images)
            loss = criterion(outputs, lbls)
            val_loss += loss.item()

            # Accuracy
            _, predicted = torch.max(outputs, 1)
            total += lbls.size(0)
            correct += (predicted == lbls).sum().item()

    val_loss = val_loss / len(val_loader)
    val_acc = correct / total

    print(f"Epoch [{epoch+1}/{EPOCHS}] | "
          f"Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | "
          f"Val Acc: {val_acc*100:.2f}%")


Epoch [1/10] | Train Loss: 0.6934 | Val Loss: 0.7523 | Val Acc: 50.00%
Epoch [2/10] | Train Loss: 0.7328 | Val Loss: 0.6915 | Val Acc: 50.00%
Epoch [3/10] | Train Loss: 0.6873 | Val Loss: 0.6918 | Val Acc: 50.00%
Epoch [4/10] | Train Loss: 0.6859 | Val Loss: 0.6934 | Val Acc: 50.00%
Epoch [5/10] | Train Loss: 0.6791 | Val Loss: 0.6960 | Val Acc: 50.00%
Epoch [6/10] | Train Loss: 0.6743 | Val Loss: 0.6991 | Val Acc: 50.00%
Epoch [7/10] | Train Loss: 0.6650 | Val Loss: 0.7000 | Val Acc: 50.00%
Epoch [8/10] | Train Loss: 0.6482 | Val Loss: 0.6984 | Val Acc: 50.00%
Epoch [9/10] | Train Loss: 0.6249 | Val Loss: 0.6966 | Val Acc: 50.00%
Epoch [10/10] | Train Loss: 0.5954 | Val Loss: 0.7057 | Val Acc: 50.00%


In [34]:
model.eval()
test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for images, lbls in test_loader:
        images, lbls = images.to(device), lbls.to(device)

        outputs = model(images)
        loss = criterion(outputs, lbls)
        test_loss += loss.item()

        # Accuracy
        _, predicted = torch.max(outputs, 1)
        total += lbls.size(0)
        correct += (predicted == lbls).sum().item()

test_loss = test_loss / len(test_loader)
test_acc = correct / total

print(f"\nTEST RESULTS")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc*100:.2f}%")



TEST RESULTS
Test Loss: 0.6464
Test Accuracy: 50.00%
